# Baseline scenario: NG boiler + thermal storage + grid import
**Master thesis (EPFL x Voltiris): *A Path to Greenhouse Self-Sufficiency: Heat and CO₂***

This notebook only **runs** the model and shows the results. The model itself lives in the `ghmodel` package (`src/ghmodel/`):

| Step | Where the code is |
|---|---|
| Parameters and Excel column mapping | `config.py` |
| Read and check the hourly Excel | `data.py` |
| Component sizes and cost factors | `sizing.py` |
| Building blocks (boiler, storage, grid, sinks) | `blocks.py` |
| Baseline network | `scenarios/baseline.py` |
| Solve, extract results, KPIs, export | `workflow.py`, `results.py` |
| Plots | `plots.py` |

Units: MW, MWh, t CO₂, EUR. Flows, assumptions and equations are explained in `README.md` and the model PDF. If the Excel file is missing, a **synthetic dummy dataset** is created so the notebook runs, its numbers mean nothing.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

# find the project root (the folder that contains src/ghmodel) so the notebook runs from anywhere
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "ghmodel").exists())
sys.path.insert(0, str(ROOT / "src"))

%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
from IPython.display import display

from ghmodel import Params, INPUT_COLUMNS, OPTIONAL_COLUMNS
from ghmodel.data import make_dummy_input, load_inputs, input_overview
from ghmodel.sizing import resolve_sizes
from ghmodel.scenarios import baseline
from ghmodel.workflow import solve
from ghmodel.results import extract_results, check_balances, summarise, export_results
from ghmodel.analysis import storage_value, financial_analysis
from ghmodel import plots

pd.options.display.float_format = "{:,.2f}".format

## 2. Configuration: files and plot options
The Excel column headers and unit factors are defined in `src/ghmodel/config.py` (`INPUT_COLUMNS`). To use other headers, edit them there or override them here, for example `INPUT_COLUMNS["heat_demand_MW"] = ("my_heat_column", 1e-3)`.

In [ ]:
INPUT_FILE  = ROOT / "data" / "baseline_input_data.xlsx"   # put your real data here
SHEET_NAME  = "hourly"
TIME_COL    = "timestamp"                  # set None if the sheet has no datetime column
START_TIMESTAMP = "2023-01-01 00:00"       # used only if TIME_COL is None / not found
GENERATE_DUMMY_IF_MISSING = True

RESULTS_DIR  = ROOT / "results" / "baseline"
SAVE_FIGURES = True
FIG_DPI      = 150
ZOOM_WINTER_START  = None     # e.g. "2023-01-16"; None = coldest week
ZOOM_SUMMER_START  = None     # e.g. "2023-07-10"; None = week with lowest heat demand
ZOOM_STORAGE_START = None     # e.g. "2023-05-08"; None = week with highest storage use

FIG_DIR = RESULTS_DIR / "figures" if SAVE_FIGURES else None
plots.apply_style(FIG_DPI)

## 3. Model parameters
Values marked *placeholder* in `config.py` are typical orders of magnitude and **must be replaced with sourced data**. `None` means "size automatically from the data". Only the values you want to change need to be listed, everything else keeps the default from `config.py`. All parameters are shown here for convenience.

In [ ]:
from ghmodel.config import COUNTRY_DEFAULTS

# ---- Country: sets the default carbon levy on gas and the grid emission factor
COUNTRY = "CH"                            # CH / FR / NL

# Country values (placeholders, defaults from config.py). Edit here to change them for this run only,
# or set co2_price_EUR_per_t / grid_co2_factor_t_per_MWh in Params below to override the country value.
COUNTRY_DEFAULTS["CH"].update(co2_levy_EUR_per_t=125.0, grid_ef_t_per_MWh=0.10)
COUNTRY_DEFAULTS["FR"].update(co2_levy_EUR_per_t=44.6,  grid_ef_t_per_MWh=0.05)
COUNTRY_DEFAULTS["NL"].update(co2_levy_EUR_per_t=0.0,   grid_ef_t_per_MWh=0.30)

display(pd.DataFrame(COUNTRY_DEFAULTS).T.rename(columns={"co2_levy_EUR_per_t": "Carbon levy [EUR/t CO2]",
                                                         "grid_ef_t_per_MWh": "Grid emission factor [t CO2/MWh]"}))

In [ ]:
P = Params(
    # ---- Greenhouse
    greenhouse_area_m2 = 10_000,
    country            = COUNTRY,         # set in the cell above

    # ---- Natural gas
    gas_co2_factor_t_per_MWh = 0.202,     # t CO2 per MWh fuel (LHV)
    co2_price_EUR_per_t      = None,      # carbon levy on gas (EUR/t); None -> country value above

    # ---- Conventional boiler
    boiler_eta                 = 0.92,    # LHV, placeholder
    boiler_capacity_MW_th      = None,    # None -> boiler_sizing_margin x peak heat demand
    boiler_sizing_margin       = 1.2,
    boiler_co2_usable_fraction = 1.0,     # placeholder
    boiler_capex_EUR_per_kW_th = 80,      # placeholder
    boiler_lifetime_y          = 20,
    boiler_fom_frac            = 0.02,
    boiler_ramp_limit_per_h    = 0.5,     # max output change per hour, share of capacity; None = off, placeholder

    # ---- Thermal storage
    tes_capacity_MWh         = None,      # None -> tes_hours_of_peak x peak heat demand
    tes_hours_of_peak        = 6,
    tes_power_MW_th          = None,      # None -> boiler thermal capacity
    tes_hours_at_full_power  = 4.0,       # charge/discharge power <= capacity / hours; None = off, placeholder
    tes_min_soc_frac         = 0.1,       # reserve, share of capacity never used; 0 = off, placeholder
    tes_eta_charge           = 0.98,      # placeholder
    tes_eta_discharge        = 0.98,      # placeholder
    tes_standing_loss_per_h  = 0.001,     # placeholder
    tes_capex_EUR_per_kWh    = 10,        # placeholder
    tes_lifetime_y           = 25,
    tes_fom_frac             = 0.01,

    # ---- Grid (import only)
    grid_import_limit_MW           = None,
    grid_tariff_adder_EUR_per_MWh  = 0.0,
    grid_co2_factor_t_per_MWh      = None,   # None -> country value above

    # ---- Economics
    discount_rate      = 0.05,
    project_lifetime_y = 20,              # horizon of the financial analysis (section 10)

    # ---- Size optimisation: False = fixed input, True = decision variable
    OPTIMISE_BOILER      = False,
    OPTIMISE_TES         = False,
    tes_capacity_max_MWh = None,          # optional upper bound when OPTIMISE_TES = True

    # ---- Assumed sinks (not drawn in the figure): 0 = free
    heat_dump_cost_EUR_per_MWh = 0.0,
    co2_vent_cost_EUR_per_t    = 0.0,

    # ---- Slack (numerical safety, should stay at 0)
    voll_heat_EUR_per_MWh = 1_000,
    voll_co2_EUR_per_t    = 5_000,

    solver_name = "highs",                # solver_options: see config.py (HiGHS defaults, 1 h time limit)
)
print(f"Country {P.country}: carbon levy {P.co2_levy():.1f} EUR/t CO2, grid emission factor {P.grid_ef():.3f} t CO2/MWh")

## 4. Input data

In [ ]:
if not INPUT_FILE.exists():
    if GENERATE_DUMMY_IF_MISSING:
        make_dummy_input(INPUT_FILE, SHEET_NAME)
        print(f"'{INPUT_FILE.name}' not found -> synthetic dummy file created. Replace it with real data.")
    else:
        raise FileNotFoundError(INPUT_FILE)

data = load_inputs(INPUT_FILE, sheet_name=SHEET_NAME, time_col=TIME_COL, start_timestamp=START_TIMESTAMP,
                   params=P, input_columns=INPUT_COLUMNS, optional_columns=OPTIONAL_COLUMNS)
print(f"{len(data)} hourly values, {data.index[0]} -> {data.index[-1]}")
display(input_overview(data, P.greenhouse_area_m2))

In [ ]:
plots.plot_inputs(data, out_dir=FIG_DIR)

## 5. Build the network

In [ ]:
sizes = resolve_sizes(data, P)
display(sizes.table())
print(f"Boiler size optimised: {P.OPTIMISE_BOILER} | Storage size optimised: {P.OPTIMISE_TES}")

n = baseline.build(data, P, sizes)
print(n)

## 6. Solve: minimise operating cost

In [ ]:
solve(n, P)

## 7. Results and KPIs
Hourly results `r` have all flows positive (MW, t/h, MWh, EUR). The balance check must pass and the unserved demand must be zero.

In [ ]:
r = extract_results(n, data, P)
check_balances(r)
res = summarise(n, r, data, P, sizes)

display(res.summary)
display(res.heat_balance.to_frame())

In [ ]:
display(res.monthly)

## 8. Plots

### 8.1 Heat: yearly view

In [ ]:
plots.plot_heat_yearly(res, out_dir=FIG_DIR)

### 8.2 Three hourly weeks

In [ ]:
plots.plot_zoom_weeks(res, zoom_winter_start=ZOOM_WINTER_START, zoom_summer_start=ZOOM_SUMMER_START,
                      zoom_storage_start=ZOOM_STORAGE_START, out_dir=FIG_DIR)

### 8.3 Thermal storage

In [ ]:
plots.plot_storage(res, out_dir=FIG_DIR)

### 8.4 CO₂

In [ ]:
plots.plot_co2(res, out_dir=FIG_DIR)

### 8.5 Fuel, grid and monthly cost

In [ ]:
plots.plot_fuel_grid_cost(res, out_dir=FIG_DIR)

### 8.6 Duration curves

In [ ]:
plots.plot_duration(res, out_dir=FIG_DIR)

### 8.7 Heatmaps

In [ ]:
plots.plot_heatmaps(res, out_dir=FIG_DIR)

### 8.8 Cost breakdown and fate of the boiler heat

In [ ]:
plots.plot_cost_and_heat_fate(res, out_dir=FIG_DIR)

## 9. Value of the thermal storage
The same scenario is solved a second time **without storage** (capacity and power set to 0). Everything else is unchanged, including whether the boiler size is optimised. **Saved by storage = cost without storage − cost with storage**, so a positive value means the storage saves money.

- With fixed sizes, the saving is purely operational: less gas, less levy, and less heat dumped, because CO2-driven daytime heat is shifted to the night. The storage capex is then subtracted to get the net saving.
- With `OPTIMISE_BOILER = True`, the saving also includes the smaller boiler that the storage allows.
- Simple payback = net investment / net yearly operating saving, with no discounting.

In [ ]:
tes_table, tes_indicators, res_no_tes = storage_value(baseline.build, data, P, res)
display(tes_table)
display(tes_indicators)
plots.plot_storage_savings(tes_table, tes_indicators, out_dir=FIG_DIR)

## 10. Financial analysis
The equipment of the Baseline (the greenhouse itself is excluded) is the **boiler** and the **thermal storage**. The boiler has no alternative inside this scenario, since it is the only heat and CO2 source, so its payback is not defined here. The S1 to S4 notebooks compute the payback of their whole equipment set against the Baseline. In the Baseline, the investment that can be judged is the **storage**: the reference is the same system without storage (section 9).

Cash flows over `project_lifetime_y` years:
- **Year 0:** the additional investment.
- **Every year:** the operating cost saving minus the additional fixed O&M.
- **Replacements** when a lifetime ends inside the horizon, and a **straight-line salvage value** at the end.

Outputs: simple and discounted payback, NPV, IRR, CO2 abatement cost, and a first-order NPV sensitivity (±20 % on prices and investment, dispatch not re-optimised).

In [ ]:
fin = financial_analysis(res, res_no_tes, P, name="Baseline", ref_name="Baseline without storage")
print("Equipment of the Baseline (overnight investment, lifetime, fixed O&M):")
display(fin["investments"])
display(fin["indicators"])
display(fin["sensitivity"])
plots.plot_financials(fin, out_dir=FIG_DIR)

## 11. Export

In [ ]:
export_results(res, n, RESULTS_DIR, name="baseline")